# SentinelFlow AI — CIC-IDS2017 Data Preprocessing

This notebook builds the preprocessing pipeline for the CIC-IDS2017 dataset based on the data quality issues identified during the initial exploration phase.

## Preprocessing Objectives

- Normalize column names
- Resolve label encoding inconsistencies
- Handle missing values
- Handle positive and negative infinity values
- Remove duplicate network flows
- Preserve the original raw dataset
- Produce a clean processed dataset for machine learning

## Guiding Principle

All transformations are applied to copies of the raw data. The original files in `data/raw/` remain unchanged.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Project paths
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

# Ensure the processed-data directory exists
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Discover raw CIC-IDS2017 CSV files
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

print("Preprocessing environment ready.")
print(f"Raw data directory: {RAW_DATA_DIR.resolve()}")
print(f"Processed data directory: {PROCESSED_DATA_DIR.resolve()}")
print(f"CSV files found: {len(csv_files)}")

Preprocessing environment ready.
Raw data directory: C:\Users\abdal\SentinelFlow-AI\data\raw
Processed data directory: C:\Users\abdal\SentinelFlow-AI\data\processed
CSV files found: 8


## 1. Inspect Raw Labels

Before applying transformations, inspect the raw traffic labels to identify formatting or encoding inconsistencies.

In [2]:
raw_labels = set()

for file in csv_files:
    labels = pd.read_csv(
        file,
        usecols=lambda col: col.strip() == "Label"
    )

    labels.columns = labels.columns.str.strip()
    raw_labels.update(labels["Label"].dropna().unique())

print(f"Unique raw labels: {len(raw_labels)}\n")

for label in sorted(raw_labels):
    print(repr(label))

Unique raw labels: 15

'BENIGN'
'Bot'
'DDoS'
'DoS GoldenEye'
'DoS Hulk'
'DoS Slowhttptest'
'DoS slowloris'
'FTP-Patator'
'Heartbleed'
'Infiltration'
'PortScan'
'SSH-Patator'
'Web Attack � Brute Force'
'Web Attack � Sql Injection'
'Web Attack � XSS'


## 2. Define Cleaning Rules

Standardize column names and traffic labels before applying further data-quality transformations.

In [3]:
def clean_dataframe(df):
    """
    Apply basic schema and label normalization to a CIC-IDS2017 DataFrame.
    """
    
    df = df.copy()

    # Remove leading/trailing whitespace from column names
    df.columns = df.columns.str.strip()

    # Normalize traffic labels
    if "Label" in df.columns:
        df["Label"] = (
            df["Label"]
            .astype("string")
            .str.strip()
            .str.replace("�", "", regex=False)
            .str.replace(r"\s+", " ", regex=True)
        )

    return df

In [4]:
sample_df = pd.read_csv(csv_files[0], nrows=5)

clean_sample = clean_dataframe(sample_df)

print(f"Columns before cleaning: {len(sample_df.columns)}")
print(f"Columns after cleaning:  {len(clean_sample.columns)}")
print(f"Target column found:      {'Label' in clean_sample.columns}")
print(f"Original data unchanged:  {' Label' in sample_df.columns}")

Columns before cleaning: 79
Columns after cleaning:  79
Target column found:      True
Original data unchanged:  True


In [5]:
cleaned_labels = set()

for file in csv_files:
    labels = pd.read_csv(
        file,
        usecols=lambda col: col.strip() == "Label"
    )

    labels = clean_dataframe(labels)

    cleaned_labels.update(
        labels["Label"].dropna().unique()
    )

print(f"Unique cleaned labels: {len(cleaned_labels)}\n")

for label in sorted(cleaned_labels):
    print(repr(label))

Unique cleaned labels: 15

'BENIGN'
'Bot'
'DDoS'
'DoS GoldenEye'
'DoS Hulk'
'DoS Slowhttptest'
'DoS slowloris'
'FTP-Patator'
'Heartbleed'
'Infiltration'
'PortScan'
'SSH-Patator'
'Web Attack Brute Force'
'Web Attack Sql Injection'
'Web Attack XSS'


## 3. Handle Missing and Infinite Values

The initial exploration showed that invalid numeric values are limited to `Flow Bytes/s` and `Flow Packets/s`.

Positive and negative infinity values are first converted to missing values so that all invalid numeric values can be handled consistently.

In [6]:
def normalize_invalid_values(df):
    """
    Replace positive and negative infinity values with NaN.
    """
    df = df.copy()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df

In [7]:
test_df = pd.read_csv(csv_files[0], low_memory=False)
test_df = clean_dataframe(test_df)

before_invalid = test_df[["Flow Bytes/s", "Flow Packets/s"]].isna().sum()

test_cleaned = normalize_invalid_values(test_df)

after_invalid = test_cleaned[["Flow Bytes/s", "Flow Packets/s"]].isna().sum()

print("Before converting infinity:")
print(before_invalid)

print("\nAfter converting infinity:")
print(after_invalid)

Before converting infinity:
Flow Bytes/s      4
Flow Packets/s    0
dtype: int64

After converting infinity:
Flow Bytes/s      34
Flow Packets/s    34
dtype: int64


### Invalid Row Impact

Before deciding how to handle missing values, the number of network flows containing at least one invalid value is measured.

This avoids making preprocessing decisions based only on the number of invalid cells.

In [9]:
invalid_row_results = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)

    df = clean_dataframe(df)
    df = normalize_invalid_values(df)

    invalid_mask = df.isna().any(axis=1)
    invalid_rows = int(invalid_mask.sum())

    invalid_row_results.append({
        "file": file.name,
        "rows": len(df),
        "invalid_rows": invalid_rows,
        "invalid_percentage": (invalid_rows / len(df)) * 100
    })

    del df

invalid_rows_df = pd.DataFrame(invalid_row_results)

invalid_rows_df

,file,rows,invalid_rows,invalid_percentage
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,34,0.015061
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,371,0.129509
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,122,0.063863
3,Monday-WorkingHours.pcap_ISCX.csv,529918,437,0.082466
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,207,0.071725
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,135,0.079241
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,264,0.059205
7,Wednesday-workingHours.pcap_ISCX.csv,692703,1297,0.187238


In [10]:
total_invalid_rows = invalid_rows_df["invalid_rows"].sum()
total_rows = invalid_rows_df["rows"].sum()

print(f"Total invalid rows: {total_invalid_rows:,}")
print(f"Percentage of dataset affected: {(total_invalid_rows / total_rows) * 100:.4f}%")

Total invalid rows: 2,867
Percentage of dataset affected: 0.1013%


### Invalid Value Handling Decision

Only **2,867 network flows (0.1013%)** contain at least one missing or infinite value.

Because the affected portion of the dataset is extremely small, these rows are removed rather than imputed. This avoids introducing synthetic values into network-traffic features while preserving approximately **99.90%** of the original observations.

The preprocessing pipeline therefore:

1. Converts positive and negative infinity values to `NaN`.
2. Removes rows containing any remaining missing values.

In [11]:
def remove_invalid_rows(df):
    """
    Convert infinite numeric values to NaN and remove rows
    containing missing or invalid values.
    """
    df = df.copy()

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    rows_before = len(df)

    df.dropna(inplace=True)

    rows_removed = rows_before - len(df)

    return df, rows_removed

In [12]:
test_df = pd.read_csv(csv_files[0], low_memory=False)
test_df = clean_dataframe(test_df)

rows_before = len(test_df)

test_cleaned, rows_removed = remove_invalid_rows(test_df)

print(f"Rows before:  {rows_before:,}")
print(f"Rows removed: {rows_removed:,}")
print(f"Rows after:   {len(test_cleaned):,}")

print(f"\nRemaining missing values: {test_cleaned.isna().sum().sum():,}")

numeric_test = test_cleaned.select_dtypes(include=[np.number])

print(
    "Remaining infinite values:",
    np.isinf(numeric_test.to_numpy()).sum()
)

Rows before:  225,745
Rows removed: 34
Rows after:   225,711

Remaining missing values: 0
Remaining infinite values: 0


## Duplicate Network Flows

The exploratory analysis identified a substantial number of duplicate network flows.

Before removing duplicates, their class distribution is analyzed to determine whether deduplication could disproportionately affect benign or malicious traffic classes.

In [13]:
duplicate_label_frames = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)

    df = clean_dataframe(df)
    df, _ = remove_invalid_rows(df)

    duplicate_mask = df.duplicated(keep="first")

    duplicate_labels = (
        df.loc[duplicate_mask, "Label"]
        .value_counts()
        .rename_axis("traffic_class")
        .reset_index(name="duplicate_rows")
    )

    duplicate_label_frames.append(duplicate_labels)

    del df

duplicate_distribution = (
    pd.concat(duplicate_label_frames, ignore_index=True)
    .groupby("traffic_class", as_index=False)["duplicate_rows"]
    .sum()
    .sort_values("duplicate_rows", ascending=False)
)

duplicate_distribution["percentage_of_duplicates"] = (
    duplicate_distribution["duplicate_rows"]
    / duplicate_distribution["duplicate_rows"].sum()
    * 100
)

duplicate_distribution

,traffic_class,duplicate_rows,percentage_of_duplicates
0,BENIGN,124421,48.747434
8,PortScan,68110,26.685107
4,DoS Hulk,57278,22.441192
9,SSH-Patator,2678,1.049225
7,FTP-Patator,2004,0.785156
6,DoS slowloris,411,0.161027
5,DoS Slowhttptest,271,0.106176
10,Web Attack Brute Force,37,0.014496
2,DDoS,11,0.00431
1,Bot,8,0.003134


### Impact of Duplicate Removal on Class Distribution

Duplicate flows are not uniformly distributed across traffic classes. Most duplicates belong to BENIGN, PortScan, and DoS Hulk traffic.

Before applying deduplication to the final processed dataset, the class distribution before and after duplicate removal is compared to measure its impact.

In [14]:
class_impact_frames = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)

    df = clean_dataframe(df)
    df, _ = remove_invalid_rows(df)

    before = df["Label"].value_counts()

    deduplicated_df = df.drop_duplicates()
    after = deduplicated_df["Label"].value_counts()

    comparison = pd.DataFrame({
        "before": before,
        "after": after
    }).fillna(0)

    comparison["removed"] = comparison["before"] - comparison["after"]

    class_impact_frames.append(comparison)

    del df, deduplicated_df

class_impact = (
    pd.concat(class_impact_frames)
    .groupby(level=0)
    .sum()
)

class_impact["removed_percentage"] = (
    class_impact["removed"]
    / class_impact["before"]
    * 100
)

class_impact = (
    class_impact
    .sort_values("before", ascending=False)
    .reset_index()
    .rename(columns={"Label": "traffic_class", "index": "traffic_class"})
)

class_impact

,traffic_class,before,after,removed,removed_percentage
0,BENIGN,2271320,2146899,124421,5.477916
1,DoS Hulk,230124,172846,57278,24.890059
2,PortScan,158804,90694,68110,42.889348
3,DDoS,128025,128014,11,0.008592
4,DoS GoldenEye,10293,10286,7,0.068007
5,FTP-Patator,7935,5931,2004,25.255198
6,SSH-Patator,5897,3219,2678,45.412922
7,DoS slowloris,5796,5385,411,7.091097
8,DoS Slowhttptest,5499,5228,271,4.928169
9,Bot,1956,1948,8,0.408998


### Duplicate Removal Decision

Duplicate network flows are not uniformly distributed across traffic classes.

The largest proportional reductions occur in:

- SSH-Patator: ~45.41%
- PortScan: ~42.89%
- FTP-Patator: ~25.26%
- DoS Hulk: ~24.89%
- BENIGN: ~5.48%

Despite this uneven distribution, exact duplicate rows are removed from the processed dataset. Keeping identical observations could artificially increase the influence of repeated network flows during model training and may lead to overly optimistic evaluation results.

The resulting class imbalance will be measured after preprocessing and handled explicitly during the machine learning stage rather than by retaining exact duplicate observations.

In [15]:
def remove_duplicate_rows(df):
    """
    Remove exact duplicate network flows while keeping
    the first occurrence.
    """
    df = df.copy()

    rows_before = len(df)

    df.drop_duplicates(keep="first", inplace=True)

    rows_removed = rows_before - len(df)

    return df, rows_removed

In [16]:
test_df = pd.read_csv(csv_files[0], low_memory=False)

test_df = clean_dataframe(test_df)
test_df, invalid_removed = remove_invalid_rows(test_df)

rows_before_duplicates = len(test_df)

test_deduplicated, duplicates_removed = remove_duplicate_rows(test_df)

print(f"Rows before duplicate removal: {rows_before_duplicates:,}")
print(f"Duplicate rows removed:        {duplicates_removed:,}")
print(f"Rows after duplicate removal:  {len(test_deduplicated):,}")

print(
    f"Remaining duplicate rows:      "
    f"{test_deduplicated.duplicated().sum():,}"
)

Rows before duplicate removal: 225,711
Duplicate rows removed:        2,629
Rows after duplicate removal:  223,082
Remaining duplicate rows:      0


## Build the Processed Dataset

The validated preprocessing steps are now applied to all eight CIC-IDS2017 CSV files.

For each file, the pipeline:

1. Normalizes column names.
2. Cleans inconsistent traffic labels.
3. Converts infinite values to missing values.
4. Removes rows containing invalid values.
5. Removes exact duplicate network flows.
6. Saves the cleaned data to `data/processed/`.

The original files in `data/raw/` remain unchanged.

In [18]:
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

# Create processed directory if it does not already exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(RAW_DIR.glob("*.csv"))

print(f"Raw directory: {RAW_DIR.resolve()}")
print(f"Processed directory: {PROCESSED_DIR.resolve()}")
print(f"CSV files found: {len(csv_files)}")

Raw directory: C:\Users\abdal\SentinelFlow-AI\data\raw
Processed directory: C:\Users\abdal\SentinelFlow-AI\data\processed
CSV files found: 8


In [19]:
processing_results = []

for file in csv_files:
    print(f"Processing: {file.name}")

    df = pd.read_csv(file, low_memory=False)

    original_rows = len(df)

    # 1. Clean column names and labels
    df = clean_dataframe(df)

    # 2. Remove invalid rows
    df, invalid_removed = remove_invalid_rows(df)

    # 3. Remove exact duplicates
    df, duplicates_removed = remove_duplicate_rows(df)

    # 4. Save processed file
    output_file = PROCESSED_DIR / file.name
    df.to_csv(output_file, index=False)

    processing_results.append({
        "file": file.name,
        "original_rows": original_rows,
        "invalid_removed": invalid_removed,
        "duplicates_removed": duplicates_removed,
        "final_rows": len(df)
    })

    print(
        f"  {original_rows:,} -> {len(df):,} rows "
        f"(invalid: {invalid_removed:,}, duplicates: {duplicates_removed:,})"
    )

    del df

processing_summary = pd.DataFrame(processing_results)

processing_summary

Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  225,745 -> 223,082 rows (invalid: 34, duplicates: 2,629)
Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  286,467 -> 213,777 rows (invalid: 371, duplicates: 72,319)
Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
  191,033 -> 184,044 rows (invalid: 122, duplicates: 6,867)
Processing: Monday-WorkingHours.pcap_ISCX.csv
  529,918 -> 502,650 rows (invalid: 437, duplicates: 26,831)
Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  288,602 -> 252,790 rows (invalid: 207, duplicates: 35,605)
Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  170,366 -> 164,179 rows (invalid: 135, duplicates: 6,052)
Processing: Tuesday-WorkingHours.pcap_ISCX.csv
  445,909 -> 421,626 rows (invalid: 264, duplicates: 24,019)
Processing: Wednesday-workingHours.pcap_ISCX.csv
  692,703 -> 610,492 rows (invalid: 1,297, duplicates: 80,914)


,file,original_rows,invalid_removed,duplicates_removed,final_rows
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,34,2629,223082
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,371,72319,213777
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,122,6867,184044
3,Monday-WorkingHours.pcap_ISCX.csv,529918,437,26831,502650
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,207,35605,252790
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,135,6052,164179
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,264,24019,421626
7,Wednesday-workingHours.pcap_ISCX.csv,692703,1297,80914,610492


## Final Validation

The processed dataset is validated after saving to disk to ensure that the preprocessing pipeline produced clean and consistent output.

The validation checks:

- All expected processed CSV files exist
- Each file contains 79 columns
- The target `Label` column is present
- No missing values remain
- No positive or negative infinity values remain
- No exact duplicate rows remain
- The final number of network flows is recorded

In [20]:
processed_files = sorted(PROCESSED_DIR.glob("*.csv"))

print(f"Processed files found: {len(processed_files)}")

validation_results = []

for file in processed_files:
    print(f"Validating: {file.name}")

    df = pd.read_csv(file, low_memory=False)

    numeric_df = df.select_dtypes(include=[np.number])

    missing_values = int(df.isna().sum().sum())
    infinite_values = int(np.isinf(numeric_df.to_numpy()).sum())
    duplicate_rows = int(df.duplicated().sum())

    validation_results.append({
        "file": file.name,
        "rows": len(df),
        "columns": len(df.columns),
        "label_present": "Label" in df.columns,
        "missing_values": missing_values,
        "infinite_values": infinite_values,
        "duplicate_rows": duplicate_rows
    })

    del df, numeric_df

validation_df = pd.DataFrame(validation_results)

validation_df

Processed files found: 8
Validating: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Validating: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Validating: Friday-WorkingHours-Morning.pcap_ISCX.csv
Validating: Monday-WorkingHours.pcap_ISCX.csv
Validating: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Validating: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Validating: Tuesday-WorkingHours.pcap_ISCX.csv
Validating: Wednesday-workingHours.pcap_ISCX.csv


,file,rows,columns,label_present,missing_values,infinite_values,duplicate_rows
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,223082,79,True,0,0,0
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,213777,79,True,0,0,0
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,184044,79,True,0,0,0
3,Monday-WorkingHours.pcap_ISCX.csv,502650,79,True,0,0,0
4,Thursday-WorkingHours-Afternoon-Infilteration....,252790,79,True,0,0,0
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,164179,79,True,0,0,0
6,Tuesday-WorkingHours.pcap_ISCX.csv,421626,79,True,0,0,0
7,Wednesday-workingHours.pcap_ISCX.csv,610492,79,True,0,0,0


In [21]:
total_processed_rows = validation_df["rows"].sum()

print(f"Processed files: {len(validation_df)}")
print(f"Total processed rows: {total_processed_rows:,}")

print("\nValidation checks:")
print(f"All files have 79 columns: {(validation_df['columns'] == 79).all()}")
print(f"Label present in all files: {validation_df['label_present'].all()}")
print(f"Total missing values: {validation_df['missing_values'].sum():,}")
print(f"Total infinite values: {validation_df['infinite_values'].sum():,}")
print(f"Total duplicate rows: {validation_df['duplicate_rows'].sum():,}")

Processed files: 8
Total processed rows: 2,572,640

Validation checks:
All files have 79 columns: True
Label present in all files: True
Total missing values: 0
Total infinite values: 0
Total duplicate rows: 0
